# Multi-Token Prediction

Analyzes whether and how models represent future tokens beyond the immediate next token. Tests for planning behavior, lookahead in the residual stream, and future information encoded at each layer.

**References:**
- Gloeckle et al. (2024) "Better & Faster Large Language Models via Multi-token Prediction"
- Pal et al. (2023) "Future Lens: Anticipating Subsequent Tokens"

## Why This Matters

Multi-token prediction analysis probes whether models plan ahead — can intermediate representations predict not just the next token but tokens further in the future? Planning horizon and next-k accuracy measurements reveal whether models implement genuine lookahead or process one token at a time.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.multi_token_prediction import (
    future_token_probing,
    planning_horizon,
    next_k_token_accuracy,
    representation_lookahead,
    future_information_by_layer,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")
print(f"Input: {len(tokens)} tokens")

## 1. Future Token Probing

Use the unembedding matrix to check if the representation at position i contains information about the token at position i + offset. If the model "plans ahead," future tokens should have low rank in the residual stream.

In [ ]:
# Probe for the next token (offset=1)
probe1 = future_token_probing(model, tokens, future_offset=1, layer=-1)
print("Next-token probing (offset=1):")
print(f"  Mean rank: {probe1['mean_rank']:.1f} (out of {cfg.d_vocab})")
print(f"  Mean prob: {probe1['mean_prob']:.4f}")
print(f"  Top-10 fraction: {probe1['top_10_fraction']:.4f}")

# Probe for token 2 positions ahead
probe2 = future_token_probing(model, tokens, future_offset=2, layer=-1)
print(f"\nOffset-2 probing:")
print(f"  Mean rank: {probe2['mean_rank']:.1f}")
print(f"  Mean prob: {probe2['mean_prob']:.4f}")
print(f"  Top-10 fraction: {probe2['top_10_fraction']:.4f}")

## 2. Planning Horizon

Estimate how many tokens ahead the model effectively "plans" by probing at increasing offsets. The effective horizon is the furthest offset where prediction is still better than chance.

In [ ]:
horizon = planning_horizon(model, tokens, max_lookahead=5)

print("Planning horizon analysis:")
for k in range(len(horizon['offset_probs'])):
    print(f"  Offset {k+1}: mean_prob={horizon['offset_probs'][k]:.4f}, "
          f"mean_rank={horizon['offset_ranks'][k]:.1f}")

print(f"\nEffective horizon: {horizon['effective_horizon']} tokens")
print(f"Horizon decay rate: {horizon['horizon_decay_rate']:.4f}")

## 3. Next-k Token Accuracy

Measure top-1 and top-5 accuracy at predicting k tokens ahead from the model's output logits at each position.

In [ ]:
accuracy = next_k_token_accuracy(model, tokens, k=4)

print("Next-k token accuracy:")
for k in range(4):
    print(f"  Offset {k+1}: top-1={accuracy['top_1_accuracy_per_offset'][k]:.4f}, "
          f"top-5={accuracy['top_5_accuracy_per_offset'][k]:.4f}")

print(f"\nMean top-1 accuracy: {accuracy['mean_top_1']:.4f}")
print(f"Best offset: {accuracy['best_offset']}")

## 4. Representation Lookahead

How much future context is encoded in the current representation? Measures cosine similarity between representations at nearby positions.

In [ ]:
lookahead = representation_lookahead(model, tokens, pos=-1, layer=-1)

print(f"Similarity to nearby positions: {lookahead['similarity_to_future']}")
print(f"Mean similarity: {lookahead['mean_future_similarity']:.4f}")
print(f"Weighted lookahead distance: {lookahead['lookahead_distance']:.2f}")
print(f"Max similarity offset: {lookahead['max_similarity_offset']}")

## 5. Future Information by Layer

Probe for the next+offset token at each layer to track when future information enters the residual stream. Early emergence suggests the model builds lookahead representations early.

In [ ]:
by_layer = future_information_by_layer(model, tokens, future_offset=1)

print("Future token predictability by layer:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: mean_prob={by_layer['layer_mean_probs'][l]:.4f}, "
          f"mean_rank={by_layer['layer_mean_ranks'][l]:.1f}")

print(f"\nBest layer for future prediction: {by_layer['best_layer']}")
print(f"Emergence layer (rank < d_vocab/5): {by_layer['emergence_layer']}")